# 05 — Model mathematics and numerical interpretation

This supplement explains and checks the **unchanged original MLForge package**.
Read notebooks 01–04 first for the original workflow. We add no estimator and
use scikit-learn only for data and utilities. See [dataset contracts](../docs/DATASETS.md).

**Questions:** What objectives do the models solve? What do the coefficients
mean? How do conditioning and the gradient connect mathematics to executable code?

The diabetes experiment uses only the original training rows for its fitted
summaries. The logistic derivative check uses small synthetic data and the public
`fit` API, not a new private helper. The original package's extreme-range numerical
limitations remain; see [findings](../docs/FINDINGS_AND_LIMITATIONS.md).


In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from mlforge import LinearRegression

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False})
assert LinearRegression.__module__ == "mlforge.linear_regression"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## 1. Load raw features and split the data

scaled=False avoids the loader's complete-dataset normalization.
We make a reproducible 80/20 split before computing preprocessing
statistics. All ten features are retained; the test set is not used
for feature selection or model fitting.

In [2]:
dataset = load_diabetes(as_frame=True, scaled=False)
X, y = dataset.data, dataset.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
assert set(X_train.index).isdisjoint(X_test.index)
display(pd.Series({
    "Total observations": len(X),
    "Features": X.shape[1],
    "Training observations": len(X_train),
    "Test observations": len(X_test),
}, name="Dataset and split").to_frame())

,Dataset and split
Total observations,442
Features,10
Training observations,353
Test observations,89


## 2. Standardize using training data only

Least squares with an intercept does not require standardization,
but scaling makes differently sized feature columns easier to work
with and improves numerical conditioning. Fit the scaler only on
training rows and apply the same transformation to test rows.
Keep the target in its original units.

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
assert scaler.n_samples_seen_ == len(X_train)
np.testing.assert_allclose(scaler.mean_, X_train.mean().to_numpy())
np.testing.assert_allclose(X_train_scaled.mean(axis=0), 0.0, atol=1e-12)

### Mathematical model and implementation

For $n$ observations and $p$ features, $X\in\mathbb{R}^{n\times p}$ and $y\in\mathbb{R}^n$. Include an intercept with $A=[\mathbf{1},X]\in\mathbb{R}^{n\times(p+1)}$ and $\theta=(b,w^T)^T$. Predictions are $\hat y=A\theta$.

We minimize
$$J(\theta)=\frac{1}{2n}\|A\theta-y\|_2^2,\qquad
\nabla J(\theta)=\frac1n A^T(A\theta-y).$$
Setting the gradient to zero gives the normal equations $A^TA\hat\theta=A^Ty$. Full column rank gives unique coefficients. A rank-deficient design can have multiple coefficient vectors with the same fitted values; `lstsq` chooses the minimum-norm solution.

We use `numpy.linalg.lstsq` rather than explicitly forming $(A^TA)^{-1}$. Forming the normal-equation matrix can worsen conditioning; an inverse is unnecessary. The factor $1/(2n)$ changes the gradient scale, not the least-squares minimizer.

For $x=(0,1,2,3)$ and $y=(1,3,5,7)$, the solution is $b=1,w=2$, with zero residuals. The package's analytical tests verify this case. More generally, a least-squares training residual satisfies $A^T(y-A\hat\theta)\approx0$ up to floating-point error.

**Implementation:** `src/mlforge/linear_regression.py`; mathematical checks in `tests/test_linear_regression.py` and `tests/test_numerical_contracts.py`. Textbook: Chapter 12 (PDF pp. 102–109).

## 3. Fit the custom model on training data

MLForge fits an intercept and coefficients by minimizing squared errors
with NumPy's least-squares solver. Scikit-learn is used for utilities,
not for the regression model.

In [4]:
model = LinearRegression(fit_intercept=True).fit(X_train_scaled, y_train)

### Coefficients, conditioning, and training fit

A coefficient below measures target-unit change per one **training standard deviation** of that feature, holding the other encoded features fixed. The intercept is the prediction at the vector of training feature means. The coded sex variable is discrete, so a one-SD change is an algebraic scale, not a literal continuous intervention.

Correlated features can have unstable individual coefficients even when predictions are more stable. These are conditional associations, not causal effects. Rank and condition number describe this design numerically; neither proves population validity.

In [5]:
coefficient_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient per training SD": model.coef_,
}).set_index("Feature")
display(coefficient_table.round(4))
print(f"Intercept: {model.intercept_:.4f}")
A = np.column_stack((np.ones(len(X_train_scaled)), X_train_scaled))
A_raw = np.column_stack((np.ones(len(X_train)), X_train.to_numpy()))
print(f"Augmented design rank: {np.linalg.matrix_rank(A)} / {A.shape[1]}")
print(f"Condition number: raw={np.linalg.cond(A_raw):.2f}, standardized={np.linalg.cond(A):.2f}")
training_predictions = model.predict(X_train_scaled)
training_residuals = y_train.to_numpy() - training_predictions
np.testing.assert_allclose(A.T @ training_residuals, 0., atol=1e-7)
print(f"Training R²: {model.score(X_train_scaled, y_train):.4f}")
print(f"Training RMSE: {np.sqrt(mean_squared_error(y_train, training_predictions)):.4f}")

Intercept: 153.7365
Augmented design rank: 11 / 11
Condition number: raw=7065.49, standardized=20.73
Training R²: 0.5279
Training RMSE: 53.5588


,Coefficient per training SD
Feature,
age,1.7538
sex,-11.5118
bmi,25.6071
bp,16.8289
s1,-44.4489
s2,24.6410
s3,7.6770
s4,13.1388
s5,35.1612


### Interpretation of the regression calculation

The design has rank 11 out of 11. Standardization reduces its observed condition
number from approximately 7065 to 21. This is evidence of improved numerical
conditioning, not improved held-out predictive accuracy by itself. With a full-rank
unregularized affine model, invertible feature rescaling preserves the set of
possible predictions. Rank-deficient minimum-norm coefficient choices require
more care because the norm depends on coordinates.

The intercept is approximately 153.7365, the fitted response at the training
feature means. The standardized BMI coefficient is approximately 25.6071 target
units per training SD, conditional on the other features. Correlated serum measures
make individual coefficients difficult to interpret. No coefficient establishes
causality. Training R² of 0.5279 and RMSE of 53.5588 measure in-sample fit only.


## 4. From a Bernoulli model to gradient descent

For $n$ observations, use $A=[\mathbf1,X]$ and $\theta=(b,w^T)^T$. The logit is $z=A\theta$ and $p=\sigma(z)=1/(1+e^{-z})$. Here $p$ means **P(benign)** because the package target is 1 for benign and 0 for malignant.

Assuming conditionally independent Bernoulli labels, the likelihood is $\prod_i p_i^{y_i}(1-p_i)^{1-y_i}$. Taking the negative log and averaging gives
$$J(\theta)=-\frac1n\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)].$$
The chain rule with $\sigma'(z)=\sigma(z)(1-\sigma(z))$ yields
$$\frac{\partial\ell_i}{\partial z_i}=p_i-y_i,\qquad
\nabla J(\theta)=A^T(p-y)/n,\qquad
\theta_{t+1}=\theta_t-\eta\nabla J(\theta_t).$$

The code averages before its matrix product and uses separate sigmoid branches to avoid exponential overflow. For binary labels it evaluates the equivalent stable loss `mean(logaddexp(0, (1-2*y)*z))`. Tests check the gradient used by training against central finite differences at nonzero parameter vectors.

The objective is convex, but 2,000 updates do not certify convergence. On perfectly separable data, unregularized coefficients can grow without a finite optimum. A small training loss also does not establish generalization or calibration. Scaling helps the learning rate operate across features; the learning rate changes optimization, not the model family.

**Implementation:** `src/mlforge/logistic_regression.py`; tests in `tests/test_logistic_regression.py` and `tests/test_numerical_contracts.py`. Textbook: Chapters 13–14 and 16.

## 5. Checking the actual public training update

Write the gradient as $g(\theta)=A^T(\sigma(A\theta)-y)/n$.
The original estimator does not expose a gradient method. Instead, fit once for
one update and once for two updates, both from its deterministic zero initialization.
Then $(\theta_1-\theta_2)/\eta$ recovers the gradient used at the nonzero
$\theta_1$. Compare it with central finite differences of the independently
written objective. This checks actual training behavior without editing source.

The check uses moderate logits, float tolerances and a small finite-difference step;
it is evidence at these inputs, not a proof for all floating-point values.


In [6]:
from mlforge import LogisticRegression
X_small = np.array([[1., -2.], [0., 1.], [-1., .5], [2., 1.]])
y_small = np.array([0., 1., 0., 1.])
A_small = np.column_stack((np.ones(len(X_small)), X_small))
eta = .2
one = LogisticRegression(learning_rate=eta, max_iter=1).fit(X_small, y_small)
two = LogisticRegression(learning_rate=eta, max_iter=2).fit(X_small, y_small)
theta1 = np.r_[one.intercept_, one.coef_]
theta2 = np.r_[two.intercept_, two.coef_]
observed_gradient = (theta1-theta2)/eta

def objective(theta):
    logits = A_small @ theta
    return np.mean(np.logaddexp(0., logits) - y_small*logits)

h = 1e-5
numeric_gradient = np.empty_like(theta1)
for j in range(len(theta1)):
    shift = np.zeros_like(theta1)
    shift[j] = h
    numeric_gradient[j] = (objective(theta1+shift)-objective(theta1-shift))/(2*h)
np.testing.assert_allclose(observed_gradient, numeric_gradient, atol=1e-9, rtol=1e-6)
display(pd.DataFrame({"Actual update gradient": observed_gradient,
                      "Finite-difference gradient": numeric_gradient}))
print("Maximum absolute difference:", np.max(np.abs(observed_gradient-numeric_gradient)))
print("Objective at theta1 / theta2:", objective(theta1), objective(theta2))

Maximum absolute difference: 1.7687518116815681e-12
Objective at theta1 / theta2: 0.6441913237250027 0.6019656509903286


,Actual update gradient,Finite-difference gradient
0,0.008957,0.008957
1,-0.234043,-0.234043
2,-0.404941,-0.404941


## Mathematical conclusions

Least squares solves a convex quadratic; its normal equations characterize
residual orthogonality. A stable least-squares solver avoids explicitly inverting
$A^TA$. Logistic regression minimizes convex binary cross-entropy, but fixed-step
iterations need not have converged. On separable data, finite maximum-likelihood
coefficients may not exist; decreasing loss does not imply bounded weights.

Keep the original implementations and notebook results distinct from universal
numerical guarantees. Neither scaling nor these tests repairs the known original
Linear Regression overflow behavior. Chapter connections in the supplied textbook:
12 (least squares), 13 (gradient descent), 14 (classification).
